In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import numpy as np
import sys

from taurex.log import disableLogging

import taurex
taurex.log.disableLogging()

from taurex.cache import OpacityCache,CIACache
OpacityCache().clear_cache()
OpacityCache().set_opacity_path("./xsec")
CIACache().set_cia_path("./cia")

from taurex.mixin import enhance_class
from taurex.planet import Planet
from taurex.stellar import PhoenixStar, BlackbodyStar

from taurex.data.profiles.pressure import SimplePressureProfile
from taurex.temperature import Guillot2010, NPoint, Isothermal
from taurex.constants import AU, RJUP

from taurex.temperature import TemperatureFile
from taurex.data.profiles.chemistry import TaurexChemistry
from taurex.chemistry import ConstantGas

from taurex.contributions import AbsorptionContribution
from taurex.contributions import CIAContribution
from taurex.contributions import RayleighContribution

from explor.model import HotSpotPhaseCurveModel
import astropy

import pandas as pd
import numpy as np
import os
import xarray as xr
from taurex.binning import FluxBinner
from taurex.util.util import create_grid_res
from matplotlib.pyplot import cm

In [2]:
def bindown_single(w1, d1, w2, d2, output, noise, eclipses):
    wf = (w1 + w2) / 2
    df = (w2 + d2/2) - (w1 - d1/2)

    photogrid = np.array([[wf], [df]])

    fb2 = FluxBinner(photogrid[0], photogrid[1])

    # returns wavelength, value, and error
    wl, val, err, *_ = fb2.bindown(
        output[0], output[1],
        error=noise/np.sqrt(eclipses)
    )

    return w1, w2, wl[0], val[0], err[0], df

def make_next_level_points(results):
    """
    results: list of (w1, w2, wl, val, err, df)
             length must be even
    returns: list of (w1, d1, w2, d2)
    """

    assert len(results) % 2 == 0

    new_points = []

    for i in range(0, len(results), 2):
        left_bin  = results[i]
        right_bin = results[i + 1]

        # left edge of combined bin
        w1 = left_bin[2] - left_bin[5] / 2
        # right edge of combined bin
        w2 = right_bin[2] + right_bin[5] / 2

        width = w2 - w1

        new_points.append((w1, width, w2, width))

    return new_points

def bindown_multiple(output, noise, wlgrid, fpfs, eclipses, name, *new_points):
    all_results = []

    for (w1, d1, w2, d2) in new_points:
        out_i = bindown_single(w1, d1, w2, d2, output, noise, eclipses)
        all_results.append(out_i)

    return all_results

def create_point(w1, w2):
    wb1 = wb[np.argmin(np.abs(wl - w1))]
    wb2 = wb[np.argmin(np.abs(wl - w2))]
    point = (w1, wb1, w2, wb2)
    return point

In [3]:
#load combined csv 
df = pd.read_csv("combined_nightside_results.csv", index_col=0)
df

,Planet,Surface Pressure [bar],Atmospheric Components,Hydrogen Inventory [H oceans],Redox State,Integration Time (hours),Number of observations,Observations with bb fit,Mean Temperature,Median Temperature,1-sigma Lower,1-sigma Upper
0,HD3167,10.902882,"SO2, H2O",H10,IW4,345.60,15,18.0,684.793264,899.046517,4.260118,1078.279651
1,HD3167,301.889080,"CO2, N2",H20,IW4,299.52,13,15.0,681.030145,879.790166,22.643581,1080.161661
2,K2141,169.221367,"SO2, H2O",H20,IW4,53.76,8,8.0,832.795583,1075.447271,9.345867,1330.462023
3,K2141,16.028394,"SO2, S2",H30,IW2,53.76,8,8.0,875.997260,1083.199393,410.973486,1343.706663
4,LHS1478,317.212366,"H2O, S2",H05,IW0,468000.00,10000,9850.0,232.036301,314.986889,3.034050,347.804389
5,LHS1478,2693.216197,"CO2, H2O",H05,IW4,468000.00,10000,9850.0,234.972027,317.958710,11.123128,349.009930
7,TOI1416,55.970885,"CO2, SO2",H10,IW4,1176.00,49,53.0,572.883813,766.109011,0.000000,938.521682
8,TOI1416,437.019741,"CO, N2",H20,IW0,840.00,35,37.0,596.799536,796.349503,0.000000,986.541918
9,TOI1807,153.474293,"CO, N2",H10,IW0,277.20,21,21.0,666.128679,910.418318,0.000000,1126.003234
10,TOI1807,13.566413,"SO2, H2O",H5,IW4,422.40,32,32.0,628.129859,852.666742,0.000000,1062.576214


In [4]:
planet_names = ["HD3167","K2141","LHS1478","TOI431","TOI500","TOI561","TOI1416","TOI1807"]
planet_masses = [4.73, 4.97, 2.33, 3.07, 1.42, 2.02, 3.48, 2.44] #Earth masses
planet_distances = [0.018, 0.007, 0.018, 0.011, 0.012, 0.011, 0.019, 0.012] #AU
planet_period = [0.96, 0.28, 1.95, 0.49, 0.55, 0.45, 1.0, 0.55] #days
planet_radius = [1.627, 1.510, 1.242, 1.277, 1.166, 1.397, 1.620, 1.496] #Earth radii
planet_transit = [1.61, 0.94, 0.71, 1.24, 0.99, 1.31, 1.5, 0.98] #hours
planet_transit = [n * 3600 for n in planet_transit] #hours to seconds
T_transit_hours = [1.61, 0.94, 0.71, 1.24, 0.99, 1.31, 1.5, 0.98] #hours
planet_impact = [0.181, -0.01, 0.717, 0.34, 0.53, 0.14, 0.39, 0.489] #to be fixed according to archive data
planet_eccentricity = [0.05, 0.0, 0.0, 0.0, 0.06, 0.0, 0.0, 0.0] #to be fixed according to archive data
planet_pericentre_long = [0.0, 90.0, 0.0, 0.0, 228.5, 0.0, 0.0, 90.0] #w, to be fixed according to archive data

star_temperature = [5261.0,4570.0,3381.0,4850.0,4440.0,5342.0,4884.0,4914.0] #Kelvin
star_radius = [0.872,0.681,0.246,0.731,0.678,0.856,0.793,0.746] #Solar radii
star_metallicity = [0.03, 0.0, -0.13, 0.2, 0.12, -0.4, 0.08, -0.04] #[Fe/H]
star_logg = [4.5, 4.6, 4.9, 4.6, 4.6, 4.5, 4.5, 4.6]
star_age = [10.2, 6.3, 5.6, 5.1, 5, 11, 6.9, 0.3] #Gyr
star_distance = [47.28, 61.87, 18.22, 32.6, 47.39, 85.8, 55.01, 42.58] #pc, to be fixed

name = planet_names[0]
H = '20'
iw = '4'
eff = '00001'

#optional
S = '20'

N_pc = 32

In [ ]:
mass = planet_masses[planet_names.index(name)] #Earth masses
#convert to jupiter masses
mass_jup = mass / 317.8
print(mass_jup)
#semi-major axis
a = planet_distances[planet_names.index(name)] #AU

radius = planet_radius[planet_names.index(name)] #planet radius in m

#convert to Jupiter radii
radius_jup = radius * (astropy.constants.R_earth.value / astropy.constants.R_jup.value)
print(radius_jup)


#set-up planet in Jupiter masses and radii
pl = Planet(planet_mass=mass_jup, planet_radius= radius_jup, planet_sma=a, planet_distance=star_distance[planet_names.index(name)], 
            impact_param=planet_impact[planet_names.index(name)], orbital_period=planet_period[planet_names.index(name)], 
            transit_time=planet_transit[planet_names.index(name)])

st = PhoenixStar(temperature=star_temperature[planet_names.index(name)], radius=star_radius[planet_names.index(name)], 
                 metallicity=star_metallicity[planet_names.index(name)],distance=star_distance[planet_names.index(name)], phoenix_path='Phoenix/')

In [ ]:
pl._mid_time = 0 #T_c, leave at 0
pl.pericentre_time = 0.0 # leave to 0
pl.ascending_node_long = 0.0 # leave to 0

pl._eccentricity = planet_eccentricity[planet_names.index(name)] # eccentricity
pl._pericentre_long = planet_pericentre_long[planet_names.index(name)] #w

#convert from seconds to days
transittime = pl.transitTime*1.15741e-5
_period = planet_period[planet_names.index(name)]
phase_window = 0.65  # fraction of period shown on each side (0.5 = ±half period, reduce to shorten)
begin = pl._mid_time - phase_window * _period
end   = pl._mid_time + phase_window * _period

phases = np.linspace(begin, end, 200)

In [ ]:
#PRESSURE PROFILE
# pressure profile from  planet specific csv file
if name == 'K2141' or name == 'TOI431':
    path_to_csv = f'./PLANETS/{name}/H{H}_IW{iw}_{eff}_S{S}_TP.csv'
else:
    path_to_csv = f'./PLANETS/{name}/H{H}_IW{iw}_{eff}_TP.csv'

#open the csv file and extract pressure and temperature
data = pd.read_csv(path_to_csv)
pressure = data['Pressure (Pa)'].values

p1 = SimplePressureProfile(nlayers=100, atm_min_pressure=np.min(pressure), atm_max_pressure=np.max(pressure))
p1.compute_pressure_profile()

ps = [p1,p1,p1]

In [ ]:
#TEMPREATURE PROFILE
#temperature profile from  planet specific csv file

if name == 'K2141' or name == 'TOI431':
    planetdir = f'./PLANETS/{name}/H{H}_IW{iw}_{eff}_S{S}_TP.csv'
else:
    planetdir = f'./PLANETS/{name}/H{H}_IW{iw}_{eff}_TP.csv'

tp1 = TemperatureFile(planetdir, skiprows=1,
                               temp_col=1, press_col=0,
                               temp_units='K', press_units='Pa',
                               delimiter = ',')

tp2 = TemperatureFile(planetdir, skiprows=1,
                               temp_col=2, press_col=0,
                               temp_units='K', press_units='Pa',
                               delimiter = ',')


temperatures = [tp2, tp2, tp1]

In [ ]:
for folder in os.listdir(f"./PLANETS/{name}/"):
    simulation_folder = os.path.join(f"PLANETS/{name}/", folder)
    #check if it is a directory
    if not os.path.isdir(simulation_folder):
        continue
    for file in os.listdir(f"./{simulation_folder}/"):
        if file.endswith("atm.nc"):
            atm_file = os.path.join(f"./{simulation_folder}/", file)
            #save filename without extension
            filename = os.path.splitext(atm_file)[0]
            break

    ds = xr.open_dataset(atm_file)

    #extract gas names
    gases = np.array(ds['gases'])
    gases = [m.decode().strip() for m in ds["gases"].values]
    vmr = np.array(ds['x_gas'])

    pressure = np.array(ds['p']) #pressure in Pa
    pmax = pressure.max()
    pmin = pressure.min()

    radius = float(ds['planet_radius']) #planet radius in m
    #convert to Jupiter radii
    radius = radius / astropy.constants.R_jup.value

    #get mixing ratio of each molecule
    H2O_x = float(vmr[:, gases.index('H2O')][0])
    CO2_x = float(vmr[:, gases.index('CO2')][0])
    CH4_x = float(vmr[:, gases.index('CH4')][0])
    CO_x = float(vmr[:, gases.index('CO')][0])
    NH3_x = float(vmr[:, gases.index('NH3')][0])
    N2_x = float(vmr[:, gases.index('N2')][0])
    SO2_x = float(vmr[:, gases.index('SO2')][0])
    S2_x = float(vmr[:, gases.index('S2')][0])
    O2_x = float(vmr[:, gases.index('O2')][0])
    H2_x = float(vmr[:, gases.index('H2')][0])
    H2S_x = float(vmr[:, gases.index('H2S')][0])

    #define chemistry
    chemistry = TaurexChemistry(fill_gases=["N2"])

    # check that the sum of all vmr is less or equal to 1
    total_vmr = H2O_x + CO2_x + CH4_x + CO_x + NH3_x + N2_x + SO2_x + S2_x + O2_x + H2_x + H2S_x
    if total_vmr > 1:
        #renormalize the vmr values
        H2O_x = H2O_x / total_vmr
        CO2_x = CO2_x / total_vmr
        CH4_x = CH4_x / total_vmr
        CO_x = CO_x / total_vmr
        NH3_x = NH3_x / total_vmr
        N2_x = N2_x / total_vmr
        SO2_x = SO2_x / total_vmr
        S2_x = S2_x / total_vmr
        O2_x = O2_x / total_vmr
        H2_x = H2_x / total_vmr
        H2S_x = H2S_x / total_vmr

    chemistry.addGas(ConstantGas(molecule_name="NH3", mix_ratio=NH3_x)).addGas(ConstantGas(molecule_name="CO2", mix_ratio=CO2_x)).addGas(ConstantGas(molecule_name="H2O", mix_ratio=H2O_x)).addGas(ConstantGas(molecule_name="CH4", mix_ratio=CH4_x)).addGas(ConstantGas(molecule_name="CO", mix_ratio=CO_x)).addGas(ConstantGas(molecule_name="SO2", mix_ratio=SO2_x)).addGas(ConstantGas(molecule_name="S2", mix_ratio=S2_x)).addGas(ConstantGas(molecule_name="O2", mix_ratio=O2_x)).addGas(ConstantGas(molecule_name="H2", mix_ratio=H2_x)).addGas(ConstantGas(molecule_name="H2S", mix_ratio=H2S_x))


chemistries = [chemistry, chemistry, chemistry]

In [ ]:
ctribs = [[AbsorptionContribution(), RayleighContribution(), CIAContribution(cia_pairs=['CO2-CO2','CO2-H2','CO2-H2O','H2-H2','O2-CO2'])],
                                [AbsorptionContribution(),RayleighContribution(), CIAContribution(cia_pairs=['CO2-CO2','CO2-H2','CO2-H2O','H2-H2','O2-CO2'])],
                                [AbsorptionContribution(),RayleighContribution(), CIAContribution(cia_pairs=['CO2-CO2','CO2-H2','CO2-H2O','H2-H2','O2-CO2'])]]

In [ ]:
hs = HotSpotPhaseCurveModel(phases=phases,
                 temperature_profiles=temperatures,
                 chemistry=chemistries,
                 nlayers=[70]*3,
                 pressure_profile=ps,
                 planet=pl,
                 star=st,
                 observation=None,
                 contributions=ctribs,
                 alpha_hs=45.0,
                 delta_hs = -40.0,
                 ngauss=40,
                 use_directimage=False,
                 use_cuda=False,
                 use_orbitals = True,
                 temperature_constraints = 10, res_grid=[400, 0.2, 18.5])

File already here...  teff04750_logg4.5_MH-0.1.pickle
File already here...  teff04750_logg4.5_MH0.0.pickle
File already here...  teff04750_logg5.0_MH-0.1.pickle
File already here...  teff04750_logg5.0_MH0.0.pickle
File already here...  teff05000_logg4.5_MH-0.1.pickle
File already here...  teff05000_logg4.5_MH0.0.pickle
File already here...  teff05000_logg5.0_MH-0.1.pickle
File already here...  teff05000_logg5.0_MH0.0.pickle
File already here...  teff04750_logg4.5_MH0.0.pickle
File already here...  teff04750_logg4.5_MH0.1.pickle
File already here...  teff04750_logg5.0_MH0.0.pickle
File already here...  teff04750_logg5.0_MH0.1.pickle
File already here...  teff05000_logg4.5_MH0.0.pickle
File already here...  teff05000_logg4.5_MH0.1.pickle
File already here...  teff05000_logg5.0_MH0.0.pickle
File already here...  teff05000_logg5.0_MH0.1.pickle
File already here...  teff05250_logg4.5_MH-0.3.pickle
File already here...  teff05250_logg4.5_MH-0.5.pickle
File already here...  teff05250_logg5.0_MH-0.3.pickle
File already here...  teff05250_logg5.0_MH-0.5.pickle
File already here...  teff05500_logg4.5_MH-0.3.pickle
File already here...  teff05500_logg4.5_MH-0.5.pickle
File already here...  teff05500_logg5.0_MH-0.3.pickle
File already here...  teff05500_logg5.0_MH-0.5.pickle
File already here...  teff04250_logg4.5_MH0.1.pickle
File already here...  teff04250_logg4.5_MH0.2.pickle
File already here...  teff04250_logg5.0_MH0.1.pickle
File already here...  teff04250_logg5.0_MH0.2.pickle
File already here...  teff04500_logg4.5_MH0.1.pickle
File already here...  teff04500_logg4.5_MH0.2.pickle
File already here...  teff04500_logg5.0_MH0.1.pickle
File already here...  teff04500_logg5.0_MH0.2.pickle
File already here...  teff04750_logg4.5_MH0.2.pickle
File already here...  teff04750_logg5.0_MH0.2.pickle
File already here...  teff05000_logg4.5_MH0.2.pickle
File already here...  teff05000_logg5.0_MH0.2.pickle
File already here...  teff04500_logg4.5_MH0.0.pickle
File already here...  teff04500_logg5.0_MH0.0.pickle
File already here...  teff04750_logg4.5_MH0.0.pickle
File already here...  teff04750_logg5.0_MH0.0.pickle
File already here...  teff05250_logg4.5_MH0.0.pickle
File already here...  teff05250_logg4.5_MH0.1.pickle
File already here...  teff05250_logg5.0_MH0.0.pickle
File already here...  teff05250_logg5.0_MH0.1.pickle
File already here...  teff05500_logg4.5_MH0.0.pickle
File already here...  teff05500_logg4.5_MH0.1.pickle
File already here...  teff05500_logg5.0_MH0.0.pickle
File already here...  teff05500_logg5.0_MH0.1.pickle

In [ ]:
hs.build()

In [ ]:
o = hs.model()

In [ ]:
#sanity check of temperature profile

plt.figure(figsize=(4, 3))

plt.plot(hs.temperatureProfile[0,:], hs.pressureProfile[0,:], color='green', label=f'{name}')
plt.plot(hs.temperatureProfile[1,:], hs.pressureProfile[0,:],'-.', color='green')
plt.plot(hs.temperatureProfile[2,:], hs.pressureProfile[0,:],'--', color='green')

plt.legend()
plt.yscale('log')
plt.gca().invert_yaxis()
plt.xlim(000, 4500)
plt.ylim(np.max(hs.pressureProfile), np.min(hs.pressureProfile))
plt.xlabel('Temperature (K)')
plt.ylabel('Pressure (Pa)')
#plt.savefig('all_tp.png',bbox_inches='tight', dpi=180)
plt.show()

In [ ]:
ariel = pd.read_csv(f'./ARIEL/arielrad_{name}/tier2.csv', skiprows=6)
fb_ariel = FluxBinner(ariel['Wavelength [um]'], ariel['Bandwidth [um]'])
wave = ariel['Wavelength [um]'].values
band = ariel['Bandwidth [um]'].values
error_w_floor = ariel['Noise on Transit Floor [ppm]'].values*1e-6
#error_w_hour = ariel['Total Noise [ppm]'].values*1e-6

flux = np.zeros((len(hs._orbitals), len(wave)))
fb2 = FluxBinner(wave)
for i, orb in enumerate(hs._orbitals):
    _, flux[i, :], _, _ = fb2.bindown(10000/hs.wls[0][::-1],np.array(o[1])[::-1,i])

#implement binning of error bars
new_point9 = (4.305457, 0.28225, 4.5973, 0.3014)
new_point10 = (4.908865, 0.322, 5.24157, 0.344)

new_point11 = (5.59683967, 0.367, 5.97618103, 0.391)
new_point12 = (6.38, 0.418, 6.814, 0.447)

new_points = [new_point9, new_point10, new_point11, new_point12]

In [ ]:
ariel['Noise on Transit Floor [ppm]'].values*1e-6

In [ ]:
#N_pc = 25

In [ ]:
#noise calculation
#calcualte total time
time = N_pc * planet_period[planet_names.index(name)]

dtsampled = np.mean(hs._orbitals[1:] - hs._orbitals[:-1])*24
eff_eclipses = N_pc * dtsampled / T_transit_hours[planet_names.index(name)]

In [ ]:
binned_flux = np.zeros(len(hs._orbitals))

for i in range(len(hs._orbitals)):
    output_i = (wave, flux[i, :])
    results_l1 = bindown_multiple(output_i, error_w_floor, wave, flux[i,:], eff_eclipses, name, *new_points)
    results_l2 = bindown_multiple(output_i, error_w_floor, wave, flux[i,:], eff_eclipses, name, *make_next_level_points(results_l1))
    results_l3 = bindown_multiple(output_i, error_w_floor, wave, flux[i,:], eff_eclipses, name, *make_next_level_points(results_l2))
    binned_flux[i] = results_l3[0][3]  # val

# Error is the same for all phases — read it from any phase
binned_err = results_l3[0][4]
binned_wl  = results_l3[0][2]   # central wavelength of the final bin

# Locate transit ingress using the known transit half-duration.
_idx = planet_names.index(name)

# find transit center — restrict to ±quarter-period to avoid secondary eclipse
_quarter = planet_period[_idx] / 4
_near_transit = np.abs(hs._orbitals) < _quarter
transit_center_idx = np.where(_near_transit)[0][np.argmin(binned_flux[_near_transit])]

# compute ingress index from the known transit half-duration
_transit_half = (T_transit_hours[_idx] / 2) / 24   # half-transit duration in days
_dt = np.mean(np.diff(hs._orbitals))               # step size in days

#IMPORTANT
ingress_idx = transit_center_idx - int(round(_transit_half / _dt)) - 1

#baseline from nightside (2–6 transit half-durations before ingress)
_baseline_mask = (hs._orbitals > hs._orbitals[ingress_idx] - 6 * _transit_half) &  (hs._orbitals < hs._orbitals[ingress_idx] - 2 * _transit_half)
baseline = np.median(binned_flux[_baseline_mask])

# 4 points immediately before ingress

before_ingress_idx = np.arange(ingress_idx - 2, ingress_idx+ 2)
all_idx  = np.arange(len(hs._orbitals))
main_idx = np.setdiff1d(all_idx, before_ingress_idx)
_err_night = binned_err / np.sqrt(4)

# Retry noise draw until nightside point (bottom of error bar) is fully above 1
while True:
    rand = np.random.normal(binned_flux, binned_err)
    r1, r2, r3, r4 = rand[before_ingress_idx]
    _rand_night = ((r1 + r2) / 2 + (r3 + r4) / 2) / 2
    if _rand_night - _err_night > 1:
        break

In [ ]:
plt.figure(figsize=(30, 8))
hs._orbitals_hours = hs._orbitals * 24
plt.plot(hs._orbitals_hours, binned_flux, color='black', label=f'{binned_wl:.2f} microns')
#plt.plot((hs._orbitals, hs._orbitals), (rand - binned_err, rand + binned_err), '-', color='red', alpha=0.5

plt.errorbar(hs._orbitals_hours, rand, yerr=binned_err, fmt='o', color='red', alpha=0.4)
plt.axhline(1, color='black', linestyle='--', linewidth=0.5)
plt.xlabel('Time (hours from transit)', fontsize = 14)
plt.ylabel('Normalized Flux (fractional)',fontsize = 14)
plt.title(f'Phase curve of {name}b; N={N_pc}, total time={time:.2f} days',fontsize=16)
plt.ticklabel_format(useOffset=False)
plt.tick_params(axis='both', which='major', labelsize=12)
plt.legend()
if name == 'K2141':
    plt.savefig(f'PhaseCurves/{name}_H{H}_IW{iw}_{eff}_S{S}_phase_curve.pdf',format='pdf', bbox_inches='tight')
else:
    plt.savefig(f'PhaseCurves/{name}_H{H}_IW{iw}_{eff}_phase_curve.pdf',format='pdf', bbox_inches='tight')
plt.show()

plt.figure(figsize=(30, 8))
plt.plot(hs._orbitals_hours, binned_flux, color='black', label=f'{binned_wl:.2f} microns')
#plt.plot((hs._orbitals, hs._orbitals), (rand - binned_err, rand + binned_err), '-', color='red', alpha=0.5)
plt.errorbar(hs._orbitals_hours, rand, yerr=binned_err, fmt='o', color='red', alpha=0.4)
plt.axhline(1, color='black', linestyle='--', linewidth=0.5)
plt.xlabel('Time (hours from transit)', fontsize = 14)
plt.ylabel('Normalized Flux (fractional)', fontsize = 14)
#plt.title(f'Phase curve for {name} with binned error bars (N={N_pc}, total time={time:.2f} days)')
plt.ticklabel_format(useOffset=False)
plt.tick_params(axis='both', which='major', labelsize=12)
plt.xlim(-2,0.0)
plt.legend()
if name == 'K2141':
    plt.savefig(f'PhaseCurves/{name}_H{H}_IW{iw}_{eff}_S{S}_zoomed_phase_curve.pdf',format='pdf', bbox_inches='tight')
else:
    plt.savefig(f'PhaseCurves/{name}_H{H}_IW{iw}_{eff}_zoomed_phase_curve.pdf',format='pdf', bbox_inches='tight')
plt.show()

# TIME BINNING

In [ ]:
p1, p2, p3, p4 = binned_flux[before_ingress_idx]

v_night = ((p1 + p2) / 2 + (p3 + p4) / 2) / 2
err_l1 = binned_err / np.sqrt(2)

r1, r2, r3, r4 = rand[before_ingress_idx]
rand_night = ((r1 + r2) / 2 + (r3 + r4) / 2) / 2

err_night = binned_err / np.sqrt(4)
t_night   = np.mean(hs._orbitals[before_ingress_idx])

In [ ]:
plt.figure(figsize=(30, 8))
plt.ticklabel_format(useOffset=False)

# Main phase curve
plt.plot(hs._orbitals, binned_flux, color='red', label=f'Model @ {binned_wl:.2f} µm')
plt.errorbar(hs._orbitals[main_idx], rand[main_idx], yerr=binned_err, fmt='o', color='red', alpha=0.5)

# Nightside binned point
plt.errorbar(t_night, rand_night, yerr=err_night, fmt='o', color='blue',
             capsize=5, label='Nightside (binned)',alpha=0.5)

plt.title(f'Phase curve of {name}b; N={N_pc}, total time={time:.2f} days', fontsize=16)
plt.axhline(1, color='black', linestyle='--', linewidth=0.5)
plt.xlabel('Time (days from transit)',fontsize = 14)
plt.ylabel('Normalized Flux (fractional)',fontsize = 14)
plt.tick_params(axis='both', which='major', labelsize=12)
plt.legend(loc='lower right',fontsize=15)
if name == 'K2141' or name == 'TOI431':
    plt.savefig(f'PhaseCurves/{name}_H{H}_IW{iw}_{eff}_S{S}_binned_phase_curve.pdf',format='pdf', bbox_inches='tight')
else:
    plt.savefig(f'PhaseCurves/{name}_H{H}_IW{iw}_{eff}_binned_phase_curve.pdf',format='pdf', bbox_inches='tight')
plt.show()
